In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df=pd.read_csv('telecom_churn_mock_data.csv')

In [3]:
# EDA
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerID        2000 non-null   object 
 1   Gender            2000 non-null   object 
 2   SeniorCitizen     2000 non-null   int64  
 3   Partner           2000 non-null   object 
 4   Dependents        2000 non-null   object 
 5   Tenure            2000 non-null   int64  
 6   PhoneService      2000 non-null   object 
 7   MultipleLines     2000 non-null   object 
 8   InternetService   2000 non-null   object 
 9   OnlineSecurity    2000 non-null   object 
 10  OnlineBackup      2000 non-null   object 
 11  DeviceProtection  2000 non-null   object 
 12  TechSupport       2000 non-null   object 
 13  StreamingTV       2000 non-null   object 
 14  StreamingMovies   2000 non-null   object 
 15  Contract          2000 non-null   object 
 16  PaperlessBilling  2000 non-null   object 


In [4]:
print(df.head())

  CustomerID  Gender  SeniorCitizen Partner Dependents  Tenure PhoneService  \
0   CUST1000    Male              0      No         No      30          Yes   
1   CUST1001  Female              0      No        Yes      11          Yes   
2   CUST1002  Female              1      No         No      17           No   
3   CUST1003  Female              0     Yes         No      26          Yes   
4   CUST1004    Male              0     Yes        Yes      23          Yes   

  MultipleLines InternetService       OnlineSecurity  ... DeviceProtection  \
0           Yes              No                   No  ...               No   
1           Yes     Fiber optic  No internet service  ...               No   
2            No     Fiber optic                   No  ...               No   
3            No              No                   No  ...               No   
4            No     Fiber optic  No internet service  ...               No   

           TechSupport StreamingTV      StreamingMovies 

In [5]:
print(df.isnull().sum(), '\n null count and now duplicates\n', df.duplicated().sum())

CustomerID          0
Gender              0
SeniorCitizen       0
Partner             0
Dependents          0
Tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64 
 null count and now duplicates
 0


In [6]:
duplicate_columns= df.columns[df.T.duplicated()]
print(list(duplicate_columns))
bi_col=df[['Gender','Partner','Dependents','PhoneService','MultipleLines','DeviceProtection','StreamingTV','PaperlessBilling','Churn']]
for col in bi_col:
    print(f"unique of {col}\n", bi_col[col].unique())

[]
unique of Gender
 ['Male' 'Female']
unique of Partner
 ['No' 'Yes']
unique of Dependents
 ['No' 'Yes']
unique of PhoneService
 ['Yes' 'No']
unique of MultipleLines
 ['Yes' 'No' 'No phone service']
unique of DeviceProtection
 ['No' 'Yes' 'No internet service']
unique of StreamingTV
 ['No' 'Yes' 'No internet service']
unique of PaperlessBilling
 ['No' 'Yes']
unique of Churn
 ['No' 'Yes']


In [7]:
#some of which seemed binary had more values and with mapping created nul values so i'll change them now
# so no missing values, no duplicate row nor column vise, so nothing to drop or fill
# no plot required so we can encode directly
# binary values will be converted using mapping
print(df.head())

  CustomerID  Gender  SeniorCitizen Partner Dependents  Tenure PhoneService  \
0   CUST1000    Male              0      No         No      30          Yes   
1   CUST1001  Female              0      No        Yes      11          Yes   
2   CUST1002  Female              1      No         No      17           No   
3   CUST1003  Female              0     Yes         No      26          Yes   
4   CUST1004    Male              0     Yes        Yes      23          Yes   

  MultipleLines InternetService       OnlineSecurity  ... DeviceProtection  \
0           Yes              No                   No  ...               No   
1           Yes     Fiber optic  No internet service  ...               No   
2            No     Fiber optic                   No  ...               No   
3            No              No                   No  ...               No   
4            No     Fiber optic  No internet service  ...               No   

           TechSupport StreamingTV      StreamingMovies 

In [8]:
df['Gender']= df['Gender'].map({'Male':0, 'Female': 1})
df['Partner']= df["Partner"].map({'Yes':1, 'No':0})
df['Dependents']= df["Dependents"].map({'Yes':1, 'No':0})
df['PhoneService']= df["PhoneService"].map({'Yes':1, 'No':0})
df['PaperlessBilling']= df["PaperlessBilling"].map({'Yes':1, 'No':0})
df['Churn']= df["Churn"].map({'Yes':1, 'No':0})

In [9]:
print(df.isnull().sum())

CustomerID          0
Gender              0
SeniorCitizen       0
Partner             0
Dependents          0
Tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [10]:
#now we're done with binary values we look for the unique values in each column to encode, so we can be sure no more binary
unique_of= ['PaymentMethod', 'InternetService', 'OnlineSecurity', 'TechSupport', 'StreamingMovies', 'Contract', 'OnlineBackup']
for column in unique_of:
    print(f'unique values in {column}\n', df[column].unique())

unique values in PaymentMethod
 ['Bank transfer (automatic)' 'Electronic check' 'Mailed check'
 'Credit card (automatic)']
unique values in InternetService
 ['No' 'Fiber optic' 'DSL']
unique values in OnlineSecurity
 ['No' 'No internet service' 'Yes']
unique values in TechSupport
 ['No' 'No internet service' 'Yes']
unique values in StreamingMovies
 ['No' 'No internet service' 'Yes']
unique values in Contract
 ['Month-to-month' 'Two year' 'One year']
unique values in OnlineBackup
 ['No' 'No internet service' 'Yes']


In [11]:
#let's use one hot encoding for those
df=pd.get_dummies(df, columns=['PaymentMethod', 'InternetService', 'OnlineSecurity', 'TechSupport', 'StreamingMovies', 'OnlineBackup', 'Contract', 'MultipleLines','StreamingTV', 'DeviceProtection'], dtype=int)

In [12]:
print(df.head())

  CustomerID  Gender  SeniorCitizen  Partner  Dependents  Tenure  \
0   CUST1000       0              0        0           0      30   
1   CUST1001       1              0        0           1      11   
2   CUST1002       1              1        0           0      17   
3   CUST1003       1              0        1           0      26   
4   CUST1004       0              0        1           1      23   

   PhoneService  PaperlessBilling  MonthlyCharges  TotalCharges  ...  \
0             1                 0           69.55       2047.01  ...   
1             1                 1           48.08        522.42  ...   
2             0                 0           36.56        610.07  ...   
3             1                 1           79.72       2159.26  ...   
4             1                 1           70.42       1672.56  ...   

   Contract_Two year  MultipleLines_No  MultipleLines_No phone service  \
0                  0                 0                               0   
1         

In [13]:
# last is the ID but it changes for each so we can't use mapping nor one hot encoding, so we turn it into an index instead
# otherwise it might cause overfitting cuz no generalization
df=df.set_index('CustomerID')

In [14]:
print(df.head())

            Gender  SeniorCitizen  Partner  Dependents  Tenure  PhoneService  \
CustomerID                                                                     
CUST1000         0              0        0           0      30             1   
CUST1001         1              0        0           1      11             1   
CUST1002         1              1        0           0      17             0   
CUST1003         1              0        1           0      26             1   
CUST1004         0              0        1           1      23             1   

            PaperlessBilling  MonthlyCharges  TotalCharges  Churn  ...  \
CustomerID                                                         ...   
CUST1000                   0           69.55       2047.01      0  ...   
CUST1001                   1           48.08        522.42      0  ...   
CUST1002                   0           36.56        610.07      0  ...   
CUST1003                   1           79.72       2159.26      0  ..

In [15]:
print(df.columns.tolist())

['Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes']


In [16]:
# model training so scikit learn
x = df[['PhoneService', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes','StreamingTV_No', 'StreamingTV_No internet service', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year']]
y = df['SeniorCitizen']

In [17]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [18]:
model = LogisticRegression()
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 2000 entries, CUST1000 to CUST2999
Data columns (total 41 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Gender                                   2000 non-null   int64  
 1   SeniorCitizen                            2000 non-null   int64  
 2   Partner                                  2000 non-null   int64  
 3   Dependents                               2000 non-null   int64  
 4   Tenure                                   2000 non-null   int64  
 5   PhoneService                             2000 non-null   int64  
 6   PaperlessBilling                         2000 non-null   int64  
 7   MonthlyCharges                           2000 non-null   float64
 8   TotalCharges                             2000 non-null   float64
 9   Churn                                    2000 non-null   int64  
 10  PaymentMethod_Bank transfer (automatic)  2

In [19]:
model.fit(x_train, y_train)

c:\Users\korej\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [20]:
prediction= model.predict(x_test)

In [21]:
print(f"accuracy: {accuracy_score(y_test, prediction)}\n", f"classification report: {classification_report(y_test, prediction)}" )

accuracy: 0.8175
 classification report:               precision    recall  f1-score   support

           0       0.82      1.00      0.90       327
           1       0.00      0.00      0.00        73

    accuracy                           0.82       400
   macro avg       0.41      0.50      0.45       400
weighted avg       0.67      0.82      0.74       400



c:\Users\korej\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\korej\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\korej\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
